# 6장 과제 · 직접 바꿔 보기 — 실습 1 (6_5.ipynb)

교재 과제 6개를 차례로 실행합니다. 공정하게 비교하려고 다음처럼 했습니다.

- 생성자·판별자·학습 루프는 6_5.ipynb와 같은 코드입니다. 과제에서 바꾸는 값(에폭, latent_dim, lr, BatchNorm, LeakyReLU)만 함수 인자로 꺼냈습니다.
- 모든 실험은 같은 시드(0)로 시작하고 한 번에 하나만 바꿉니다.
- 기준(원래 설정, 5 에폭)은 과제 1의 20 에폭 학습 중 5 에폭 시점을 저장해서 씁니다. 같은 학습을 두 번 하지 않기 위해서입니다.
- 손실은 교재처럼 마지막 배치 값이 아니라 에폭 평균으로 기록합니다. 판별자가 진짜에 준 평균 점수 D(x)와 가짜에 준 평균 점수 D(G(z))도 함께 기록합니다.
- 생성 그림 비교에는 같은 잡음 z를 씁니다. 그래서 그림 차이는 모델 차이입니다.
- 다양성과 모드 붕괴를 눈대중이 아니라 숫자로 보려고, 작은 MNIST 분류기로 생성 그림 1,000장이 어떤 숫자인지 세어 봅니다.

## 준비 1 · 데이터와 장치

In [ ]:
import copy, time
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

plt.rcParams["font.family"] = ["Malgun Gothic", "DejaVu Sans"]   # 그래프 제목의 한글 표시
plt.rcParams["axes.unicode_minus"] = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))    # [0,1] → [-1,1]
])
dataset = datasets.MNIST(root='data', train=True, transform=transform, download=True)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

## 준비 2 · 바꿀 수 있게 만든 생성자·판별자·학습 함수

- `Generator(latent_dim, use_bn)` — `use_bn=False`면 과제 4처럼 BatchNorm 두 줄이 빠집니다.
- `Discriminator(leaky)` — `leaky=False`면 과제 5처럼 LeakyReLU(0.2)가 ReLU()로 바뀝니다.
- `train_gan(...)` — 6_5.ipynb 셀 4·5와 같은 학습입니다. 기본값이 교재 설정(5 에폭, latent_dim 64, lr 0.0002)입니다.

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=64, use_bn=True):
        super().__init__()
        layers = [nn.ConvTranspose2d(latent_dim, 128, kernel_size=7, stride=1, padding=0)]   # → (B,128,7,7)
        if use_bn:
            layers.append(nn.BatchNorm2d(128))
        layers += [nn.ReLU(), nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1)]  # → (B,64,14,14)
        if use_bn:
            layers.append(nn.BatchNorm2d(64))
        layers += [nn.ReLU(), nn.ConvTranspose2d(64, 1, kernel_size=4, stride=2, padding=1),    # → (B,1,28,28)
                   nn.Tanh()]
        self.model = nn.Sequential(*layers)

    def forward(self, z):
        return self.model(z)


class Discriminator(nn.Module):
    def __init__(self, leaky=True):
        super().__init__()
        act = (lambda: nn.LeakyReLU(0.2)) if leaky else (lambda: nn.ReLU())
        self.model = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=4, stride=2, padding=1),     # → (B,64,14,14)
            act(),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),   # → (B,128,7,7)
            nn.BatchNorm2d(128),
            act(),
            nn.Flatten(),                                             # → (B,128*7*7)
            nn.Linear(128 * 7 * 7, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)


def train_gan(num_epochs=5, latent_dim=64, lr=0.0002, use_bn=True, leaky=True, seed=0, snapshot_epochs=()):
    torch.manual_seed(seed)
    generator = Generator(latent_dim, use_bn).to(device)
    discriminator = Discriminator(leaky).to(device)
    criterion = nn.BCELoss()
    optimizer_G = torch.optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
    optimizer_D = torch.optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))

    history = {"D": [], "G": [], "D_x": [], "D_Gz": []}
    snapshots = {}
    t0 = time.time()
    for epoch in range(num_epochs):
        generator.train()
        sums, n = {"D": 0.0, "G": 0.0, "D_x": 0.0, "D_Gz": 0.0}, 0
        for real_imgs, _ in loader:
            real_imgs = real_imgs.to(device)
            batch_size = real_imgs.size(0)
            real_labels = torch.ones(batch_size, 1).to(device)
            fake_labels = torch.zeros(batch_size, 1).to(device)

            # 1. 판별자 학습 단계
            optimizer_D.zero_grad()
            d_real = discriminator(real_imgs)
            loss_real = criterion(d_real, real_labels)
            z = torch.randn(batch_size, latent_dim, 1, 1).to(device)
            fake_imgs = generator(z)
            d_fake = discriminator(fake_imgs.detach())
            loss_fake = criterion(d_fake, fake_labels)
            loss_D = loss_real + loss_fake
            loss_D.backward()
            optimizer_D.step()

            # 2. 생성자 학습 단계
            optimizer_G.zero_grad()
            d_fake_g = discriminator(fake_imgs)
            loss_G = criterion(d_fake_g, real_labels)
            loss_G.backward()
            optimizer_G.step()

            sums["D"] += loss_D.item(); sums["G"] += loss_G.item()
            sums["D_x"] += d_real.mean().item(); sums["D_Gz"] += d_fake.mean().item()
            n += 1

        for k in history:
            history[k].append(sums[k] / n)
        print(f"Epoch {epoch+1}: D={history['D'][-1]:.4f}, G={history['G'][-1]:.4f}, "
              f"D(x)={history['D_x'][-1]:.3f}, D(G(z))={history['D_Gz'][-1]:.3f}  ({time.time()-t0:.0f}s)")
        if epoch + 1 in snapshot_epochs:
            snapshots[epoch + 1] = copy.deepcopy(generator.state_dict())
    return generator, history, snapshots

## 준비 3 · 그림 그리기 도우미와 숫자 세기용 분류기

- `fixed_z(latent_dim, n)` — 항상 같은 잡음을 돌려줍니다. 실험끼리 같은 z로 그림을 비교하기 위해서입니다.
- `classifier` — MNIST를 1 에폭 학습한 작은 CNN입니다. 생성 그림이 어떤 숫자로 보이는지 판정해, 숫자 0~9가 고르게 나오는지(다양성) 셉니다. 교재 과제에는 없고 비교를 위해 추가했습니다.

In [ ]:
def fixed_z(latent_dim, n=16, seed=123):
    g = torch.Generator().manual_seed(seed)
    return torch.randn(n, latent_dim, 1, 1, generator=g).to(device)

def make_samples(generator, latent_dim, n=16):
    generator.eval()
    with torch.no_grad():
        return generator(fixed_z(latent_dim, n)).cpu()

def show_grid(samples, title, ncol=4, ax=None):
    n = len(samples); nrow = (n + ncol - 1) // ncol
    grid = torch.ones(nrow * 29 + 1, ncol * 29 + 1) * -1
    for i in range(n):
        r, c = divmod(i, ncol)
        grid[r*29+1:r*29+29, c*29+1:c*29+29] = samples[i, 0]
    if ax is None:
        plt.figure(figsize=(ncol * 0.8, nrow * 0.8)); ax = plt.gca()
    ax.imshow(grid, cmap='gray'); ax.set_title(title); ax.axis('off')

def plot_history(histories, title):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    for name, h in histories.items():
        e = range(1, len(h["D"]) + 1)
        axes[0].plot(e, h["D"], "o-", label=f"{name} D"); axes[0].plot(e, h["G"], "s--", label=f"{name} G")
        axes[1].plot(e, h["D_x"], "o-", label=f"{name} D(x)"); axes[1].plot(e, h["D_Gz"], "s--", label=f"{name} D(G(z))")
    axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss (epoch mean)"); axes[0].legend(fontsize=8); axes[0].grid(alpha=.3)
    axes[1].set_xlabel("epoch"); axes[1].set_ylabel("D output"); axes[1].set_ylim(0, 1); axes[1].legend(fontsize=8); axes[1].grid(alpha=.3)
    fig.suptitle(title); plt.tight_layout(); plt.show()

# 숫자 세기용 분류기 (1 에폭)
torch.manual_seed(0)
classifier = nn.Sequential(
    nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Linear(128, 10)
).to(device)
opt_c = torch.optim.Adam(classifier.parameters(), lr=1e-3)
for imgs, labels in loader:
    imgs, labels = imgs.to(device), labels.to(device)
    opt_c.zero_grad(); loss = nn.functional.cross_entropy(classifier(imgs), labels); loss.backward(); opt_c.step()
test_set = datasets.MNIST(root='data', train=False, transform=transform, download=True)
classifier.eval(); correct = 0
with torch.no_grad():
    for imgs, labels in DataLoader(test_set, batch_size=500):
        correct += (classifier(imgs.to(device)).argmax(1).cpu() == labels).sum().item()
print(f"분류기 시험 정확도: {correct / len(test_set):.2%}")

def digit_counts(generator, latent_dim, n=1000):
    generator.eval()
    with torch.no_grad():
        imgs = generator(fixed_z(latent_dim, n, seed=7))
        probs = classifier(imgs).softmax(1)
    pred = probs.argmax(1).cpu()
    counts = torch.bincount(pred, minlength=10).tolist()
    sure = (probs.max(1).values > 0.9).float().mean().item()
    return counts, sure

def report_counts(rows):
    print(f"{'모델':<16}" + "".join(f"{d:>5}" for d in range(10)) + "   가장 많은 숫자 비율  분류기 확신(>0.9) 비율")
    for name, (counts, sure) in rows.items():
        print(f"{name:<16}" + "".join(f"{c:>5}" for c in counts) + f"   {max(counts)/sum(counts):>8.1%}  {sure:>14.1%}")

## 과제 1 · num_epochs = 5 → 20

같은 z 16개로 1, 5, 10, 20 에폭 시점의 그림을 비교합니다. 5 에폭 시점의 생성자는 나머지 과제의 기준(원래 설정)으로 씁니다.

In [ ]:
G20, h_base20, snaps = train_gan(num_epochs=20, snapshot_epochs=(1, 5, 10, 20))

# 기준(원래 설정, 5 에폭) 생성자와 기록
G_base = Generator(64).to(device); G_base.load_state_dict(snaps[5])
h_base = {k: v[:5] for k, v in h_base20.items()}

fig, axes = plt.subplots(1, 4, figsize=(13, 3.6))
for ax, ep in zip(axes, (1, 5, 10, 20)):
    g = Generator(64).to(device); g.load_state_dict(snaps[ep])
    show_grid(make_samples(g, 64), f"epoch {ep}", ax=ax)
plt.tight_layout(); plt.show()
plot_history({"20 epochs": h_base20}, "과제 1: 20 에폭 학습")

## 과제 2 · latent_dim 64 → 8

생성자와 학습 함수의 latent_dim을 둘 다 8로 바꿉니다 (교재 설명처럼 둘이 다르면 에러).

In [ ]:
G_z8, h_z8, _ = train_gan(num_epochs=5, latent_dim=8)

fig, axes = plt.subplots(1, 2, figsize=(10, 5.5))
show_grid(make_samples(G_base, 64, 64), "latent_dim = 64 (기준)", ncol=8, ax=axes[0])
show_grid(make_samples(G_z8, 8, 64), "latent_dim = 8", ncol=8, ax=axes[1])
plt.tight_layout(); plt.show()
report_counts({"latent 64 (기준)": digit_counts(G_base, 64), "latent 8": digit_counts(G_z8, 8)})

## 과제 3 · lr = 0.0002 → 0.002 (두 옵티마이저 모두)

In [ ]:
G_lr, h_lr, _ = train_gan(num_epochs=5, lr=0.002)

plot_history({"lr 0.0002": h_base, "lr 0.002": h_lr}, "과제 3: 학습률 10배")
fig, axes = plt.subplots(1, 2, figsize=(7, 3.6))
show_grid(make_samples(G_base, 64), "lr = 0.0002 (기준)", ax=axes[0])
show_grid(make_samples(G_lr, 64), "lr = 0.002", ax=axes[1])
plt.tight_layout(); plt.show()
report_counts({"lr 0.0002": digit_counts(G_base, 64), "lr 0.002": digit_counts(G_lr, 64)})

## 과제 4 · 생성자의 BatchNorm 두 줄 지우기

In [ ]:
G_nobn, h_nobn, _ = train_gan(num_epochs=5, use_bn=False)

plot_history({"BN 있음": h_base, "BN 없음": h_nobn}, "과제 4: 생성자 BatchNorm 제거")
fig, axes = plt.subplots(1, 2, figsize=(7, 3.6))
show_grid(make_samples(G_base, 64), "BatchNorm 있음 (기준)", ax=axes[0])
show_grid(make_samples(G_nobn, 64), "BatchNorm 없음", ax=axes[1])
plt.tight_layout(); plt.show()
report_counts({"BN 있음": digit_counts(G_base, 64), "BN 없음": digit_counts(G_nobn, 64)})

## 과제 5 · 판별자 LeakyReLU(0.2) → ReLU()

In [ ]:
G_relu, h_relu, _ = train_gan(num_epochs=5, leaky=False)

plot_history({"LeakyReLU": h_base, "ReLU": h_relu}, "과제 5: 판별자 활성화 함수")
fig, axes = plt.subplots(1, 2, figsize=(7, 3.6))
show_grid(make_samples(G_base, 64), "LeakyReLU (기준)", ax=axes[0])
show_grid(make_samples(G_relu, 64), "ReLU", ax=axes[1])
plt.tight_layout(); plt.show()
report_counts({"LeakyReLU": digit_counts(G_base, 64), "ReLU": digit_counts(G_relu, 64)})

## 과제 6 · 16장 → 64장

교재 셀 6을 `randn(64, …)`, `range(64)`, `subplot(8, 8, …)`로 바꾼 것입니다. 기준 생성자(5 에폭)와 20 에폭 생성자를 함께 봅니다.

In [ ]:
for name, g in [("기준 (5 에폭)", G_base), ("20 에폭", G20)]:
    g.eval()
    with torch.no_grad():
        z = torch.randn(64, 64, 1, 1, generator=torch.Generator().manual_seed(1)).to(device)
        samples = g(z).cpu()
    plt.figure(figsize=(6, 6))
    for i in range(64):
        plt.subplot(8, 8, i + 1)
        plt.imshow(samples[i].squeeze(), cmap='gray')
        plt.axis("off")
    plt.suptitle(name); plt.show()

report_counts({"기준 (5 에폭)": digit_counts(G_base, 64), "20 에폭": digit_counts(G20, 64)})

## 결과 정리

(실행 후 작성)